In [1]:
import csv
import warnings
from io import StringIO
from pathlib import Path

import pandas as pd



def parse_patient_dates(values: pd.Series) -> pd.Series:
    """Parse Spanish export dates and ISO dates without relying on system locale."""
    months = {
        "enero": "01", "febrero": "02", "marzo": "03",
        "abril": "04", "mayo": "05", "junio": "06",
        "julio": "07", "agosto": "08", "septiembre": "09",
        "setiembre": "09", "octubre": "10", "noviembre": "11",
        "diciembre": "12",
    }
    text = values.astype("string").str.strip().str.lower()
    text = text.str.replace("\u00e2", "", regex=False)
    text = text.str.replace(r"\s+", " ", regex=True)
    text = text.str.replace(r"([ap])\.\s*m\.", r"\1m", regex=True)
    for month, number in months.items():
        text = text.str.replace(month, number, regex=False)

    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    for date_format in (
        "%m %d, %Y, %I:%M %p", "%m %d, %Y, %I:%M:%S %p",
        "%m %d, %Y", "%Y-%m-%d", "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S.%f",
        "%Y-%m-%dT%H:%M:%S.%f",
    ):
        missing = parsed.isna() & text.notna() & text.ne("")
        parsed.loc[missing] = pd.to_datetime(
            text.loc[missing], format=date_format, errors="coerce"
        ).astype("datetime64[ns]")
    invalid = text.notna() & text.ne("") & parsed.isna()
    if invalid.any():
        warnings.warn(
            f"{values.name}: {invalid.sum()} fechas no interpretables convertidas a NaT.",
            stacklevel=2,
        )
    return parsed


def load_patient_csv(path: str | Path) -> pd.DataFrame:
    """Load the wrapped cohort export without modifying the source CSV."""
    path = Path(path)
    with path.open("r", encoding="utf-8-sig", newline="") as csv_file:
        lines = csv_file.read().splitlines()

    if not lines:
        raise ValueError(f"The CSV file is empty: {path}")

    header = next(csv.reader([lines[0]]))
    if header[:2] != ["patient_ref", "birth_date"]:
        raise ValueError("Expected patient_ref and birth_date as the first two columns.")

    records = []
    for line_number, raw_line in enumerate(lines[1:], start=2):
        if not raw_line.strip():
            continue

        stripped_line = raw_line.strip()
        fields = next(csv.reader([raw_line]))
        if (
            len(fields) == 1
            and stripped_line.startswith('"')
            and stripped_line.endswith('"')
        ):
            # The export wraps each complete row in quotes and doubles its inner quotes.
            unwrapped_line = stripped_line[1:-1].replace('""', '"')
            fields = next(csv.reader([unwrapped_line]))

        extra_fields = len(fields) - len(header)
        if extra_fields < 0:
            raise ValueError(
                f"Row {line_number} has fewer fields than the {len(header)}-column header."
            )
        if extra_fields:
            # Commas in patient_ref were not escaped, so join the split ID pieces back.
            fields = [",".join(fields[: extra_fields + 1]), *fields[extra_fields + 1 :]]

        if len(fields) != len(header):
            raise ValueError(
                f"Row {line_number} has {len(fields)} fields after repair; "
                f"expected {len(header)}."
            )
        records.append(fields)

    repaired_csv = StringIO()
    writer = csv.writer(repaired_csv)
    writer.writerow(header)
    writer.writerows(records)
    repaired_csv.seek(0)

    frame = pd.read_csv(repaired_csv)
    date_columns = [
        column for column in frame.columns
        if column.startswith("fecha_") or column.endswith("_date")
    ]
    for column in date_columns:
        frame[column] = parse_patient_dates(frame[column])
    return frame


# Support execution from the repository root or the data/ notebook directory.
PROJECT_ROOT = next(
    (root for root in (Path.cwd(), Path.cwd().parent)
     if (root / "public/info_columns_dict.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository root or data/.")


def save_patient_csv(frame: pd.DataFrame, path: Path) -> None:
    """Keep dates parsed in memory, exporting birth_date as an ISO calendar date."""
    exported = frame.copy()
    exported["birth_date"] = exported["birth_date"].dt.strftime("%Y-%m-%d").fillna("")
    exported.to_csv(path, index=False, encoding="utf-8-sig")


source_path = PROJECT_ROOT / "data/private/query_result_2026-11-05T08_34_59.csv"
df = load_patient_csv(source_path)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")

formatted_csv_path = PROJECT_ROOT / "data/private/query_result_formatted.csv"
save_patient_csv(df, formatted_csv_path)
print(f"Formatted CSV saved to: {formatted_csv_path}")


Loaded 59 rows and 32 columns.
Formatted CSV saved to: c:\Users\KSLUNA\Desktop\patient_discovery\patient-discovery\data\private\query_result_formatted.csv


In [2]:
# The public CSV is the source of truth. Validate it without overwriting it.
info_columns_dict_path = PROJECT_ROOT / "public/info_columns_dict.csv"
required_fields = [
    "column_name", "readable_name", "meaning", "section", "section_order",
    "criterion_id", "criterio", "tipo", "value_type", "explicacion",
]
with info_columns_dict_path.open(encoding="utf-8-sig", newline="") as csv_file:
    reader = csv.reader(csv_file)
    header = next(reader, [])
    if len(header) != 10 or set(header) != set(required_fields):
        raise ValueError("The dictionary must contain the ten unique expected headers.")
    for line_number, values in enumerate(reader, start=2):
        if values and len(values) != len(header):
            raise ValueError(f"Invalid dictionary field count at line {line_number}.")

column_dictionary = pd.read_csv(info_columns_dict_path, dtype=str, keep_default_na=False)
column_dictionary = column_dictionary.apply(lambda column: column.str.strip())
if column_dictionary.empty:
    raise ValueError("The dictionary cannot be empty.")
for field in ["column_name", "readable_name", "meaning", "section", "section_order"]:
    if column_dictionary[field].eq("").any():
        raise ValueError(f"Missing required dictionary values: {field}.")
if column_dictionary["column_name"].duplicated().any():
    raise ValueError("Dictionary column names must be unique.")
if not column_dictionary["section_order"].str.fullmatch(r"[+-]?\d+").all():
    raise ValueError("section_order must contain integers.")
orders = column_dictionary["section_order"].map(int)
if orders.map(abs).gt(2**53 - 1).any():
    raise ValueError("section_order exceeds JavaScript's safe integer range.")
column_dictionary["section_order"] = orders
if column_dictionary.groupby("section")["section_order"].nunique().gt(1).any():
    raise ValueError("Rows in the same section must share section_order.")
for field, allowed in {
    "tipo": {"", "Valor", "Evidencia", "Unidades"},
    "value_type": {"", "boolean", "number", "text", "date"},
}.items():
    if not column_dictionary[field].isin(allowed).all():
        raise ValueError(f"Invalid dictionary category: {field}.")

metadata = column_dictionary.set_index("column_name")
# age is derived by React, so it is intentionally absent from Python's output.
missing_descriptions = set(df.columns) - set(metadata.index)
if missing_descriptions:
    raise ValueError(f"Missing source column definitions: {sorted(missing_descriptions)}")
if "birth_date" not in df.columns or "age" in df.columns:
    raise ValueError("Python must export birth_date without calculating age.")
for name, expected_type, expected_value_type in [
    ("age", "Valor", "number"), ("birth_date", "Evidencia", "date"),
]:
    if name not in metadata.index:
        raise ValueError(f"Missing age metadata: {name}.")
    entry = metadata.loc[name]
    if (entry["tipo"], entry["value_type"], entry["criterion_id"]) != (
        expected_type, expected_value_type, "adult_age"
    ):
        raise ValueError(f"Invalid type or criterion association for {name}.")
if metadata.loc["age", "section"] != metadata.loc["birth_date", "section"]:
    raise ValueError("age and birth_date must belong to the same section.")
if not pd.api.types.is_datetime64_any_dtype(df["birth_date"]):
    raise ValueError("birth_date must remain a parsed datetime column in Python.")
adult_group = column_dictionary[
    column_dictionary["section"].eq(metadata.loc["age", "section"])
    & column_dictionary["criterion_id"].eq("adult_age")
]
if adult_group["tipo"].eq("Valor").sum() != 1:
    raise ValueError("The age criterion must have exactly one primary Valor.")

print(f"Validated {len(column_dictionary)} definitions; age is calculated in React.")


Validated 33 definitions; age is calculated in React.


In [4]:
# Export a separate copy; keep the source dataframe unchanged.
noids = df.copy()
noids["patient_ref"] = pd.factorize(noids["patient_ref"])[0] + 1
formatted_sinids_csv_path = PROJECT_ROOT / "data/query_result_formatted_noids.csv"
save_patient_csv(noids, formatted_sinids_csv_path)
print(f"Formatted without ids CSV saved to: {formatted_sinids_csv_path}")


Formatted without ids CSV saved to: c:\Users\KSLUNA\Desktop\patient_discovery\patient-discovery\data\query_result_formatted_noids.csv
